### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="5g_energy_consumption",
    dataset_year="2023",
    domain_str="technology & internet",
    # Data Source
    dataset_source="HuggingFace",
    original_dataset_source_download_link="https://huggingface.co/datasets/netop/5G-Network-Energy-Consumption",
    download_description=r"""
We use the dataset version uploaded by a top solution from the Zindi challenge.

mkdir -p local-data-warehouse/5g_energy_consumption/ \
&& wget https://github.com/ITU-AI-ML-in-5G-Challenge/5G-Energy-Consumption-Modelling-Solution-Team-Farzi-Data-Scientists/raw/refs/heads/main/ITU-5G-energy-Consumption-Dataset.zip \
&& unzip ITU-5G-energy-Consumption-Dataset.zip -d local-data-warehouse/5g_energy_consumption/ \
&& rm ITU-5G-energy-Consumption-Dataset.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{huawei_netop_5g_energy_consumption,
  author       = {{HUAWEI Netop Team}},
  title        = {5G Network Energy Consumption Dataset},
  year         = {n.d.},
  howpublished = {\url{https://huggingface.co/datasets/netop/5G-Network-Energy-Consumption}},
  note         = {Dataset hosted on Hugging Face, accessed 2026-04-15}
}
""",
    academic_reference_bibtex_key="huawei_netop_5g_energy_consumption",
    license="MIT",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
- Note: The dataset was used in a Zindi challenge, but also uploaded to Huggingface under a MIT license by the company (Huawei).
- The corresponding ITU/Zindi challenge explicitly emphasizes generalization to unseen base station products/configurations. We therefore split by base_station (BS).
- With this setup, the task is development of predictive models for network optimization where models need to generalize to new unseen base station configurations and estimate their energy consumption under similar conditions (during the same time period).
- For preprocessing, we orient on a top solution from the Zindi challenge: https://github.com/ITU-AI-ML-in-5G-Challenge/5G-Energy-Consumption-Modelling-Solution-Team-Farzi-Data-Scientists/tree/main.
- Note that the competition also used mostly samples from the known BS as the test set, but weighted unknown base stations higher in evaluation.
- The data itself is time-series. However, because we predict entirely unseen base stations in a per-sample fashion, the row-wise dependencies resulting from the temporal components cannot be used to improve performance unless the task is treated as transductive learning.
- Following the Zindi solution, we merge the three given tables and use the Cell0 information only from the cell level table. In addition, we merge the Cell1 information since it contains information as well which might be useful in a grouped split setting.
- Following the Zindi solution, we derive calendar features (`day`, `hour`, `weekday`) and drop the absolute timestamp.
- We drop constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Energy",
    problem_type="regression",
    objective_metric_name="mape",
    group_on="BS",
    group_labels="per_sample"
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
df_train = pd.read_csv(dataset_mold.path / "ECdata.csv", parse_dates=['Time'])
# df_test = pd.read_csv("/kaggle/input/ecm-itu-zindi-kp-data/imgs_202307101549519358.csv",parse_dates=['Time'])
df_cell = pd.read_csv(dataset_mold.path / "CLdata.csv",parse_dates=['Time'])
df_bs = pd.read_csv(dataset_mold.path / "BSinfo.csv")
df_features = df_cell.merge(df_bs,on=['BS','CellName'],how='outer')
df_features = df_features[df_features['CellName']=='Cell0'].reset_index(drop=True)

df_total = df_train.merge(df_features,on=['BS','Time'],how='left')

cell1 = df_cell[df_cell['CellName']=='Cell1'].rename(columns={col: col+'_Cell1' for col in df_cell.columns if col not in ['BS','Time']})
df = df_total.merge(cell1,on=['BS','Time'],how='left')

df['day'] = df['Time'].dt.day
df['weekday_number'] = df['Time'].dt.weekday
df['hour'] = df['Time'].dt.hour
df = df.drop(columns=['Time'])

# Drop constant columns that are not useful for modeling
df = df.drop(columns=['CellName', 'ESMode4', 'CellName_Cell1', 'ESMode4_Cell1', 'ESMode5_Cell1'])

cat_cols = ["BS", "RUType", "Mode"]
for col in cat_cols:
    df[col] = df[col].astype('category')

print(df.shape)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    target_feature=task_mold.target_column_name,
    problem_type=task_mold.problem_type,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for grouped data.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()